# Stage 05-C: Proposal A vs B — Comparison

Loads result JSON files produced by **05a** and **05b**, then produces:

1. **Metric table** — val & test accuracy, macro-F1, per-class F1 (Real / Fake / NEI)
2. **Training curves** — loss and macro-F1 side-by-side
3. **Side-by-side confusion matrices**
4. **Per-class F1 bar chart**
5. **Gate evolution** (Proposal B only)
6. **Summary verdict** — printed conclusion

**Prerequisites:** Run `05a_mil_attention_fusion.ipynb` and `05b_asymmetric_gated_fusion.ipynb` first.

**Outputs:**  
- `training/comparison_results/comparison_table.csv`  
- `training/comparison_results/comparison_curves.png`  
- `training/comparison_results/comparison_confusion.png`  
- `training/comparison_results/comparison_f1_bar.png`  
- `training/comparison_results/comparison_summary.json`  


In [ ]:
# ─── Environment Setup ───────────────────────────────────────────────────────
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab; return "colab", False
    except ImportError: pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"): return "vastai", False
    if Path("/workspace").exists(): return "vastai", True
    if sys.platform == "win32": return "windows", False
    if sys.platform == "darwin": return "mac", False
    return None, True

PLATFORM, _ = _detect_platform()
if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")

try: _nb_path = Path(__file__).resolve()
except NameError: _nb_path = Path.cwd()

PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection") if PLATFORM == "colab" else _nb_path.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {"colab": PROJECT_ROOT/".env.colab", "vastai": PROJECT_ROOT/".env.vastai",
            "windows": PROJECT_ROOT/".env.windows", "mac": PROJECT_ROOT/".env.mac"}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT/".env")
if not _env_file.exists():
    _env_file = PROJECT_ROOT / ".env"

from dotenv import load_dotenv; load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f"✅ DATA_ROOT: {DATA_ROOT}")

## Step 1: Configuration — point to results dirs

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()
try:
    from dotenv import load_dotenv; load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError: pass
DATA_ROOT = Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT

CONFIG = {
    # Results directories — auto-scan for the newest *_results.json in each
    "results_05a": DATA_ROOT / "training" / "stage05a_results",
    "results_05b": DATA_ROOT / "training" / "stage05b_results",
    # Output
    "output_dir":  DATA_ROOT / "training" / "comparison_results",
    # Optional: pin specific result files (None = auto-detect newest)
    "results_05a_json": None,
    "results_05b_json": None,
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
print(f"Results 05a: {CONFIG['results_05a']}")
print(f"Results 05b: {CONFIG['results_05b']}")
print(f"Output dir : {CONFIG['output_dir']}")

## Step 2: Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})
CLASS_NAMES = ["Real", "Fake", "NEI"]
print("Imports OK.")

## Step 3: Load Results JSONs

In [ ]:
def find_newest_results_json(results_dir, pinned=None):
    """Return the newest *_results.json in results_dir, or the pinned path."""
    if pinned is not None:
        p = Path(pinned)
        if not p.exists():
            raise FileNotFoundError(f"Pinned results file not found: {p}")
        return p
    d = Path(results_dir)
    if not d.exists():
        raise FileNotFoundError(
            f"Results dir not found: {d}\n"
            f"Run the corresponding notebook first."
        )
    candidates = sorted(d.glob("*_results.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(
            f"No *_results.json found in {d}\n"
            f"Run the corresponding notebook first."
        )
    return candidates[0]


path_a = find_newest_results_json(CONFIG["results_05a"], CONFIG["results_05a_json"])
path_b = find_newest_results_json(CONFIG["results_05b"], CONFIG["results_05b_json"])

with open(path_a) as f: res_a = json.load(f)
with open(path_b) as f: res_b = json.load(f)

print(f"Proposal A : {path_a.name}")
print(f"  run_name : {res_a.get('run_name')}")
print(f"  best_epoch: {res_a.get('best_epoch')}")
print()
print(f"Proposal B : {path_b.name}")
print(f"  run_name : {res_b.get('run_name')}")
print(f"  best_epoch: {res_b.get('best_epoch')}")

## Step 4: Metric Comparison Table

In [ ]:
def extract_metrics(res, split):
    """Pull val or test metrics from a results dict into a flat row."""
    key = "val_metrics" if split == "val" else "test_metrics"
    m = res.get(key, {})
    prefix = "val" if split == "val" else "test"
    return {
        "accuracy":  m.get(f"{prefix}_accuracy",  float("nan")),
        "macro_f1":  m.get(f"{prefix}_macro_f1",  float("nan")),
        "f1_real":   m.get(f"{prefix}_f1_real",   float("nan")),
        "f1_fake":   m.get(f"{prefix}_f1_fake",   float("nan")),
        "f1_nei":    m.get(f"{prefix}_f1_nei",    float("nan")),
    }


rows = []
for proposal, res, label in [
    ("A", res_a, "MIL Attention"),
    ("B", res_b, "Asym Gate + Residual"),
]:
    for split in ["val", "test"]:
        m = extract_metrics(res, split)
        rows.append({
            "Proposal": f"{proposal} — {label}",
            "Split":    split.capitalize(),
            "Accuracy": round(m["accuracy"], 4),
            "Macro F1": round(m["macro_f1"],  4),
            "F1-Real":  round(m["f1_real"],   4),
            "F1-Fake":  round(m["f1_fake"],   4),
            "F1-NEI":   round(m["f1_nei"],    4),
            "Best Ep":  res.get("best_epoch", "?"),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

csv_path = CONFIG["output_dir"] / "comparison_table.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

## Step 5: Training Curves — Loss & Macro F1 side-by-side

In [ ]:
hist_a = res_a.get("training_history", [])
hist_b = res_b.get("training_history", [])

if not hist_a or not hist_b:
    print("⚠  training_history missing in one or both results files — skipping curves.")
else:
    df_a = pd.DataFrame(hist_a)
    df_b = pd.DataFrame(hist_b)
    best_a = res_a.get("best_epoch", -1)
    best_b = res_b.get("best_epoch", -1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=False)
    fig.suptitle("Proposal A vs B — Training Curves", fontsize=13, fontweight="bold")

    # ── Loss ──
    for ax, df, label, best, color in [
        (axes[0][0], df_a, "A — MIL Attention",        best_a, "#7C3AED"),
        (axes[0][1], df_b, "B — Asym Gate + Residual", best_b, "#DC2626"),
    ]:
        ax.plot(df["epoch"], df["train_loss"], label="Train loss", color=color, alpha=0.7)
        ax.plot(df["epoch"], df["val_loss"],   label="Val loss",   color=color, linestyle="--")
        ax.axvline(best, color="gray", linestyle=":", label=f"Best ep {best}")
        ax.set_title(f"{label} — Loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("CE Loss")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    # ── Macro F1 ──
    for ax, df, label, best, color in [
        (axes[1][0], df_a, "A — MIL Attention",        best_a, "#7C3AED"),
        (axes[1][1], df_b, "B — Asym Gate + Residual", best_b, "#DC2626"),
    ]:
        ax.plot(df["epoch"], df["train_macro_f1"], label="Train", color=color, alpha=0.7)
        ax.plot(df["epoch"], df["val_macro_f1"],   label="Val",   color=color, linestyle="--")
        ax.axvline(best, color="gray", linestyle=":", label=f"Best ep {best}")
        ax.set_title(f"{label} — Macro F1")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Macro F1")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.tight_layout()
    p = CONFIG["output_dir"] / "comparison_curves.png"
    fig.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")

## Step 6: Gate Evolution — Proposal B (val gate mean per epoch)

In [ ]:
if hist_b and "avg_gate" in pd.DataFrame(hist_b).columns:
    df_b = pd.DataFrame(hist_b)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(df_b["epoch"], df_b["avg_gate"], color="#DC2626", label="val gate mean")
    ax.axhline(0.5, color="gray", linestyle=":", label="0.5 (equal weight)")
    ax.axvline(best_b, color="gray", linestyle="--", alpha=0.6, label=f"best ep {best_b}")
    ax.fill_between(df_b["epoch"], 0.5, df_b["avg_gate"],
                    where=df_b["avg_gate"] > 0.5, alpha=0.15, color="blue", label="text-dominant")
    ax.fill_between(df_b["epoch"], df_b["avg_gate"], 0.5,
                    where=df_b["avg_gate"] < 0.5, alpha=0.15, color="orange", label="image-dominant")
    ax.set_title("Proposal B — Gate Mean per Epoch (val)\n"
                 "gate > 0.5 → text wins | gate < 0.5 → image wins")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("avg gate value")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    p = CONFIG["output_dir"] / "comparison_gate_evolution.png"
    fig.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")
else:
    print("avg_gate not in Proposal B history — skipping gate plot.")

## Step 7: Side-by-side Confusion Matrices

Reconstructs confusion matrix from the per-class F1 scores and aggregate metrics stored in the results JSON.
If you want exact confusion matrices, save `y_true` / `y_pred` arrays in the results JSON and load them here.
For now we display the saved confusion matrix PNGs side-by-side.

In [ ]:
import matplotlib.image as mpimg

def find_confusion_matrix_png(results_dir, run_name):
    """Locate the saved confusion matrix PNG for a run."""
    d = Path(results_dir)
    candidates = sorted(d.glob("*_confusion_matrix.png"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        return None
    # prefer the one matching run_name
    for c in candidates:
        if run_name and run_name in c.name:
            return c
    return candidates[0]


cm_path_a = find_confusion_matrix_png(CONFIG["results_05a"], res_a.get("run_name", ""))
cm_path_b = find_confusion_matrix_png(CONFIG["results_05b"], res_b.get("run_name", ""))

if cm_path_a and cm_path_b:
    img_a = mpimg.imread(cm_path_a)
    img_b = mpimg.imread(cm_path_b)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img_a); axes[0].axis("off")
    axes[0].set_title("Proposal A — MIL Attention Fusion", fontweight="bold")
    axes[1].imshow(img_b); axes[1].axis("off")
    axes[1].set_title("Proposal B — Asym Gate + Residual", fontweight="bold")
    fig.suptitle("Test Confusion Matrices", fontsize=13, fontweight="bold")
    fig.tight_layout()
    p = CONFIG["output_dir"] / "comparison_confusion.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p}")
else:
    missing = []
    if not cm_path_a: missing.append("Proposal A confusion matrix PNG")
    if not cm_path_b: missing.append("Proposal B confusion matrix PNG")
    print(f"⚠  Missing: {missing}. Run the training notebooks first.")

## Step 8: Per-class F1 Bar Chart

In [ ]:
def get_per_class_f1(res, split):
    key = "val_metrics" if split == "val" else "test_metrics"
    m = res.get(key, {})
    p = "val" if split == "val" else "test"
    return [
        m.get(f"{p}_f1_real", float("nan")),
        m.get(f"{p}_f1_fake", float("nan")),
        m.get(f"{p}_f1_nei",  float("nan")),
    ]


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle("Per-class F1 Score — Proposal A vs B", fontsize=13, fontweight="bold")

x = np.arange(len(CLASS_NAMES))
w = 0.35
colors_a = ["#818CF8", "#4F46E5"]  # light/dark purple
colors_b = ["#FCA5A5", "#DC2626"]  # light/dark red

for ax, split, title in [(axes[0], "val", "Validation Set"), (axes[1], "test", "Test Set")]:
    f1_a = get_per_class_f1(res_a, split)
    f1_b = get_per_class_f1(res_b, split)

    bars_a = ax.bar(x - w/2, f1_a, w, label="A — MIL Attention",        color=colors_a[1], alpha=0.85)
    bars_b = ax.bar(x + w/2, f1_b, w, label="B — Asym Gate + Residual", color=colors_b[1], alpha=0.85)

    # value labels
    for bar in bars_a:
        v = bar.get_height()
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8, color=colors_a[1], fontweight="bold")
    for bar in bars_b:
        v = bar.get_height()
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8, color=colors_b[1], fontweight="bold")

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylabel("F1 Score")
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # macro F1 reference line
    macro_key = "val_macro_f1" if split == "val" else "test_macro_f1"
    mf1_a = res_a.get("val_metrics" if split=="val" else "test_metrics", {}).get(macro_key, None)
    mf1_b = res_b.get("val_metrics" if split=="val" else "test_metrics", {}).get(macro_key, None)
    if mf1_a:
        ax.axhline(mf1_a, color=colors_a[1], linestyle=":", alpha=0.5, linewidth=1)
    if mf1_b:
        ax.axhline(mf1_b, color=colors_b[1], linestyle=":", alpha=0.5, linewidth=1)

fig.tight_layout()
p = CONFIG["output_dir"] / "comparison_f1_bar.png"
fig.savefig(p, dpi=150)
plt.show()
print(f"Saved: {p}")

## Step 9: Macro F1 Overlay — both proposals on one plot

In [ ]:
if hist_a and hist_b:
    df_a = pd.DataFrame(hist_a)
    df_b = pd.DataFrame(hist_b)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(df_a["epoch"], df_a["val_macro_f1"],
            color="#7C3AED", linewidth=2, label="A val macro-F1")
    ax.plot(df_a["epoch"], df_a["train_macro_f1"],
            color="#7C3AED", linewidth=1, linestyle="--", alpha=0.5, label="A train macro-F1")
    ax.plot(df_b["epoch"], df_b["val_macro_f1"],
            color="#DC2626", linewidth=2, label="B val macro-F1")
    ax.plot(df_b["epoch"], df_b["train_macro_f1"],
            color="#DC2626", linewidth=1, linestyle="--", alpha=0.5, label="B train macro-F1")

    ax.axvline(res_a.get("best_epoch", -1), color="#7C3AED",
               linestyle=":", alpha=0.7, label=f"A best ep {res_a.get('best_epoch')}")
    ax.axvline(res_b.get("best_epoch", -1), color="#DC2626",
               linestyle=":", alpha=0.7, label=f"B best ep {res_b.get('best_epoch')}")

    ax.set_title("Val Macro F1 — Proposal A vs B (overlay)", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Macro F1")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    p = CONFIG["output_dir"] / "comparison_macro_f1_overlay.png"
    fig.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")

## Step 10: Summary Table — print & save

In [ ]:
def get_m(res, split, key):
    mk = "val_metrics" if split == "val" else "test_metrics"
    p  = "val" if split == "val" else "test"
    return res.get(mk, {}).get(f"{p}_{key}", float("nan"))


print("=" * 68)
print(f"{'Metric':<28} {'Proposal A':>18} {'Proposal B':>18}")
print("=" * 68)

METRICS = [
    ("Val Accuracy",      "val",  "accuracy"),
    ("Val Macro F1",      "val",  "macro_f1"),
    ("Val F1-Real",       "val",  "f1_real"),
    ("Val F1-Fake",       "val",  "f1_fake"),
    ("Val F1-NEI",        "val",  "f1_nei"),
    ("─" * 28,            None,   None),
    ("Test Accuracy",     "test", "accuracy"),
    ("Test Macro F1",     "test", "macro_f1"),
    ("Test F1-Real",      "test", "f1_real"),
    ("Test F1-Fake",      "test", "f1_fake"),
    ("Test F1-NEI",       "test", "f1_nei"),
    ("─" * 28,            None,   None),
    ("Best Epoch",        None,   None),
]

summary = {"proposal_a": {}, "proposal_b": {}}

for label, split, key in METRICS:
    if split is None and key is None:
        if label.startswith("─"):
            print("-" * 68)
        else:
            va = str(res_a.get("best_epoch", "?"))
            vb = str(res_b.get("best_epoch", "?"))
            print(f"{label:<28} {va:>18} {vb:>18}")
    else:
        va = get_m(res_a, split, key)
        vb = get_m(res_b, split, key)
        winner = " ◄" if (not np.isnan(va) and not np.isnan(vb) and va > vb) else ""
        winner_b = " ◄" if (not np.isnan(va) and not np.isnan(vb) and vb > va) else ""
        va_s = f"{va:.4f}{winner}" if not np.isnan(va) else "N/A"
        vb_s = f"{vb:.4f}{winner_b}" if not np.isnan(vb) else "N/A"
        print(f"{label:<28} {va_s:>18} {vb_s:>18}")
        summary["proposal_a"][f"{split}_{key}"] = va
        summary["proposal_b"][f"{split}_{key}"] = vb

print("=" * 68)
print("  ◄ = winner for that metric")

# ── auto verdict ──────────────────────────────────────────────────────────
test_f1_a = summary["proposal_a"].get("test_macro_f1", float("nan"))
test_f1_b = summary["proposal_b"].get("test_macro_f1", float("nan"))
nei_a = summary["proposal_a"].get("test_f1_nei", float("nan"))
nei_b = summary["proposal_b"].get("test_f1_nei", float("nan"))

print()
print("AUTO VERDICT")
print("-" * 68)
if not np.isnan(test_f1_a) and not np.isnan(test_f1_b):
    delta = abs(test_f1_a - test_f1_b)
    winner_overall = "A (MIL Attention)" if test_f1_a > test_f1_b else "B (Asym Gate + Residual)"
    print(f"  Overall test macro-F1 winner : {winner_overall}  (Δ = {delta:.4f})")
    if not np.isnan(nei_a) and not np.isnan(nei_b):
        winner_nei = "A" if nei_a > nei_b else "B"
        print(f"  NEI class F1 winner          : Proposal {winner_nei}  "
              f"(A={nei_a:.4f}, B={nei_b:.4f})")
        if nei_b > nei_a:
            print("  → Text residual in B appears to help NEI detection.")
        else:
            print("  → MIL attention in A better captures image evidence for NEI.")
    if delta < 0.01:
        print("  → Models are close. Prefer B for simplicity (no padding, stable).")
    elif delta < 0.02:
        print("  → Modest difference. Consider ensemble or further tuning.")
    else:
        print("  → Clear winner. Use for thesis final results.")
else:
    print("  Cannot compute verdict — test metrics missing.")

summary["verdict"] = {
    "test_macro_f1_a": test_f1_a,
    "test_macro_f1_b": test_f1_b,
    "nei_f1_a": nei_a,
    "nei_f1_b": nei_b,
    "run_name_a": res_a.get("run_name"),
    "run_name_b": res_b.get("run_name"),
    "results_json_a": str(path_a),
    "results_json_b": str(path_b),
}

p = CONFIG["output_dir"] / "comparison_summary.json"
with open(p, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: {p}")

## Step 11: Architecture Diff Table

Quick reference for the thesis table.

In [ ]:
arch_rows = [
    ("Text encoder",         "PhoBERT-base-v2 (frozen)",       "PhoBERT-base-v2 (frozen)"),
    ("Text input dim",       "768 (CLS token)",                 "768 (CLS token)"),
    ("Image encoder",        "COOLANT image_aligned_clip",      "COOLANT image_aligned_clip"),
    ("Image input dim",      "64 per pair",                     "64 (mean-pooled)"),
    ("Image aggregation",    "MIL Attention (Luong tanh-dot)",  "Mean pool (at cache time)"),
    ("Padding needed",       "Yes (max_pairs=8, zero-pad)",     "No"),
    ("Projection",           "Linear→256 + LayerNorm + ReLU",  "Linear→256 + LayerNorm + ReLU"),
    ("Fusion",               "AsymmetricGate (512→256)",        "AsymmetricGate (512→256)"),
    ("Text residual",        "None",                            "LayerNorm(h_t) added post-gate"),
    ("Classifier",           "Dropout(0.3) + Linear(256→3)",   "Dropout(0.3) + Linear(256→3)"),
    ("Output classes",       "3  (Real / Fake / NEI)",          "3  (Real / Fake / NEI)"),
    ("Trainable params",     "~200K",                           "~200K"),
    ("Side output",          "attn_weights [B, 8] per epoch",   "gate_vals [B, 256] per epoch"),
    ("Optimizer",            "AdamW lr=3e-4",                   "AdamW lr=3e-4"),
    ("Scheduler",            "OneCycleLR",                      "OneCycleLR"),
    ("Batch size",           "32 articles",                     "32 articles"),
]

df_arch = pd.DataFrame(arch_rows, columns=["Property", "Proposal A", "Proposal B"])

# highlight differences
def _highlight_diff(row):
    if row["Proposal A"] != row["Proposal B"]:
        return ["background-color: #FEF3C7"] * len(row)
    return [""] * len(row)

print(df_arch.to_string(index=False))
print("\n(Yellow rows = differs between proposals)")

arch_csv = CONFIG["output_dir"] / "architecture_diff_table.csv"
df_arch.to_csv(arch_csv, index=False)
print(f"Saved: {arch_csv}")

## All outputs

| File | Content |
|---|---|
| `comparison_table.csv` | Val + test metrics for both proposals |
| `comparison_curves.png` | 2×2 loss + macro-F1 training curves |
| `comparison_gate_evolution.png` | Proposal B gate mean per epoch |
| `comparison_confusion.png` | Side-by-side test confusion matrices |
| `comparison_f1_bar.png` | Per-class F1 grouped bar chart |
| `comparison_macro_f1_overlay.png` | Both proposals on one F1 plot |
| `comparison_summary.json` | Numeric summary + auto verdict |
| `architecture_diff_table.csv` | Architecture property diff |
